<a href="https://colab.research.google.com/github/tamannamakkarresearch-commits/Multimodel-alzheimers-detection/blob/main/train.multimodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Concatenate, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("1. Generating Multimodal Alzheimer's Benchmark Dataset (n=600)...")
n_samples = 600
# Labels: 0 = Cognitively Normal (CN), 1 = Pre-Symptomatic / Early MCI
labels = np.random.choice([0, 1], size=n_samples, p=[0.5, 0.5])

# --- MODALITY 1: CLINICAL ASSESSMENTS (MMSE, CDR, Age, nWBV) ---
mmse = np.where(labels == 0, np.random.normal(29.0, 1.0, n_samples), np.random.normal(26.5, 1.8, n_samples))
cdr = np.where(labels == 0, np.random.choice([0.0, 0.5], n_samples, p=[0.9, 0.1]), np.random.choice([0.0, 0.5], n_samples, p=[0.4, 0.6]))
age = np.random.normal(72.0, 6.5, n_samples)
nwbv = np.where(labels == 0, np.random.normal(0.76, 0.03, n_samples), np.random.normal(0.72, 0.035, n_samples))
clinical_data = np.column_stack([mmse, cdr, age, nwbv])

# --- MODALITY 2: GENETIC & FLUID BIOMARKERS (APOE-e4 alleles, Aβ42/Tau ratio) ---
apoe4 = np.where(labels == 0, np.random.choice([0, 1, 2], n_samples, p=[0.75, 0.22, 0.03]), np.random.choice([0, 1, 2], n_samples, p=[0.35, 0.45, 0.20]))
abeta_tau_ratio = np.where(labels == 0, np.random.normal(1.45, 0.25, n_samples), np.random.normal(0.95, 0.30, n_samples))
genetic_data = np.column_stack([apoe4, abeta_tau_ratio])

# Combined Tabular Data (Clinical + Genetic)
tabular_all = np.hstack([clinical_data, genetic_data])

# --- MODALITY 3: STRUCTURAL NEUROIMAGING (Simulated 32x32 T1-MRI Slices) ---
# Simulates subtle hippocampal intensity atrophy in pre-symptomatic cases
mri_images = np.random.normal(0.3, 0.1, (n_samples, 32, 32, 1))
for i in range(n_samples):
    # Create central brain/hippocampal structure
    mri_images[i, 8:24, 8:24, 0] += 0.4
    if labels[i] == 1:
        # Subtle medial temporal atrophy signature
        mri_images[i, 13:19, 13:19, 0] -= 0.18
mri_images = np.clip(mri_images, 0, 1)

# Train / Test Split (80% Train, 20% Test)
X_img_train, X_img_test, X_clin_train, X_clin_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    mri_images, clinical_data, tabular_all, labels, test_size=0.20, random_state=42, stratify=labels
)

# Normalize tabular features
scaler_clin = StandardScaler()
X_clin_train = scaler_clin.fit_transform(X_clin_train)
X_clin_test = scaler_clin.transform(X_clin_test)

scaler_tab = StandardScaler()
X_tab_train = scaler_tab.fit_transform(X_tab_train)
X_tab_test = scaler_tab.transform(X_tab_test)

# Helper function to evaluate models
results_list = []
roc_curves = {}

def evaluate_and_log(model_name, y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred) * 100
    rec = recall_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred) * 100
    auc = roc_auc_score(y_true, y_prob)
    results_list.append({
        "Model Configuration": model_name,
        "Accuracy (%)": round(acc, 2),
        "Precision (%)": round(prec, 2),
        "Recall (%)": round(rec, 2),
        "F1-Score (%)": round(f1, 2),
        "ROC-AUC": round(auc, 3)
    })
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_curves[model_name] = (fpr, tpr, auc)
    return y_pred

print("2. Running Ablation Study Models...")

# --- ABLATION 1: MRI-ONLY (2D CNN) ---
img_in1 = Input(shape=(32, 32, 1))
x = Conv2D(16, (3, 3), activation='relu', padding='same')(img_in1)
x = MaxPooling2D((2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = MaxPooling2D((2, 2))(x)
x = Flatten()(x)
x = Dense(32, activation='relu')(x)
out1 = Dense(1, activation='sigmoid')(x)
model_mri = Model(inputs=img_in1, outputs=out1)
model_mri.compile(optimizer='adam', loss='binary_crossentropy')
model_mri.fit(X_img_train, y_train, epochs=20, batch_size=32, verbose=0)
prob_mri = model_mri.predict(X_img_test, verbose=0).ravel()
evaluate_and_log("1. Unimodal: MRI-Only (CNN)", y_test, prob_mri)

# --- ABLATION 2: CLINICAL-ONLY (MLP) ---
clin_in = Input(shape=(4,))
c = Dense(16, activation='relu')(clin_in)
c = Dense(8, activation='relu')(c)
out2 = Dense(1, activation='sigmoid')(c)
model_clin = Model(inputs=clin_in, outputs=out2)
model_clin.compile(optimizer='adam', loss='binary_crossentropy')
model_clin.fit(X_clin_train, y_train, epochs=25, batch_size=32, verbose=0)
prob_clin = model_clin.predict(X_clin_test, verbose=0).ravel()
evaluate_and_log("2. Unimodal: Clinical-Only (MLP)", y_test, prob_clin)

# --- ABLATION 3: CLINICAL + GENETIC (MLP) ---
tab_in1 = Input(shape=(6,))
t = Dense(24, activation='relu')(tab_in1)
t = Dense(12, activation='relu')(t)
out3 = Dense(1, activation='sigmoid')(t)
model_tab = Model(inputs=tab_in1, outputs=out3)
model_tab.compile(optimizer='adam', loss='binary_crossentropy')
model_tab.fit(X_tab_train, y_train, epochs=25, batch_size=32, verbose=0)
prob_tab = model_tab.predict(X_tab_test, verbose=0).ravel()
evaluate_and_log("3. Bimodal: Clinical + Genetic (MLP)", y_test, prob_tab)

# --- PROPOSED MODEL: FULL MULTIMODAL FUSION (CNN + MLP) ---
# Imaging Branch
mri_input = Input(shape=(32, 32, 1), name="MRI_Input")
m = Conv2D(16, (3, 3), activation='relu', padding='same')(mri_input)
m = BatchNormalization()(m)
m = MaxPooling2D((2, 2))(m)
m = Conv2D(32, (3, 3), activation='relu', padding='same')(m)
m = MaxPooling2D((2, 2))(m)
m = Flatten()(m)
m = Dense(32, activation='relu')(m)

# Clinical + Genetic Branch
tabular_input = Input(shape=(6,), name="Clinical_Genetic_Input")
g = Dense(24, activation='relu')(tabular_input)
g = BatchNormalization()(g)
g = Dense(16, activation='relu')(g)

# Feature Fusion Layer
fused = Concatenate(name="Multimodal_Fusion_Layer")([m, g])
f = Dense(32, activation='relu')(fused)
f = Dropout(0.3)(f)
final_out = Dense(1, activation='sigmoid', name="Diagnostic_Output")(f)

multimodal_model = Model(inputs=[mri_input, tabular_input], outputs=final_out)
multimodal_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
history = multimodal_model.fit([X_img_train, X_tab_train], y_train, validation_split=0.15, epochs=30, batch_size=32, verbose=0)

prob_multi = multimodal_model.predict([X_img_test, X_tab_test], verbose=0).ravel()
pred_multi = evaluate_and_log("4. Proposed: Multimodal Fusion (CNN+MLP)", y_test, prob_multi)

# --- DISPLAY ABLATION STUDY TABLE ---
results_df = pd.DataFrame(results_list)
print("\n================ ABLATION STUDY & PERFORMANCE TABLE ================")
print(results_df.to_string(index=False))
print("====================================================================\n")

# --- GENERATE PUBLICATION PLOTS ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot 1: Comparative ROC-AUC Curves
colors = ['#8c969e', '#4c72b0', '#dd8452', '#c44e52']
for (name, (fpr, tpr, auc_val)), color in zip(roc_curves.items(), colors):
    lw = 3.0 if "Proposed" in name else 1.8
    ls = '-' if "Proposed" in name else '--'
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=color, linewidth=lw, linestyle=ls)

axes[0].plot([0, 1], [0, 1], 'k:', linewidth=1.2, label="Random Chance (AUC = 0.500)")
axes[0].set_title("Comparative ROC Curves (Ablation Study)", fontsize=13, fontweight='bold')
axes[0].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
axes[0].set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
axes[0].legend(loc="lower right", fontsize=9.5)
axes[0].grid(alpha=0.3)

# Plot 2: Confusion Matrix for Proposed Multimodal Model
cm = confusion_matrix(y_test, pred_multi)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[1],
            xticklabels=["Cognitively Normal (CN)", "Pre-Symptomatic / MCI"],
            yticklabels=["Cognitively Normal (CN)", "Pre-Symptomatic / MCI"],
            annot_kws={"size": 14, "weight": "bold"})
axes[1].set_title("Confusion Matrix: Proposed Multimodal Model", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Predicted Diagnosis", fontsize=11)
axes[1].set_ylabel("Actual Clinical Status", fontsize=11)

plt.tight_layout()
plt.savefig("Multimodal_Alzheimers_Results.png", dpi=300)
plt.show()
print("Saved high-resolution figure as 'Multimodal_Alzheimers_Results.png'!")


1. Generating Multimodal Alzheimer's Benchmark Dataset (n=600)...
2. Running Ablation Study Models...
